In [ ]:
# Cel (22.08.2026, ustalone z Maćkiem): korelacja pochodnej LOWESS z sila
# efektu byla SILNA i spojna (19/22 stacji dodatnia) przy P_days=3350, ale
# PRAWIE ZNIKALA/mieszala znak przy P_days=1675 (patrz 20260822f.ipynb).
# Pytanie: czy to realny efekt fizyczny, ktory po prostu slabnie przy
# krotszym oknie, czy artefakt zwiazany z tym, ze P_days=3350 (~9.17 roku)
# jest bliskie dlugosci calego cyklu slonecznego (~11 lat)? Test: policzyc
# ta sama korelacje dla SIATKI dodatkowych wartosci P_days - 3 pomiedzy
# 1675 a 3350 (2100, 2515, 2930) i 3 POZA dotychczasowym zakresem, w strone
# pelnej dlugosci cyklu (3600, 3800, 4000) - zeby zobaczyc KSZTALT przejscia
# (ostry prog vs stopniowy wzrost vs plateau/spadek po 3350).
#
# Zawezenie do 5 stacji reprezentatywnych (mosc, oulu, athn, hrms, sopb) -
# wszystkie juz pokazaly silna dodatnia korelacje przy P=3350 (patrz tabela
# w 20260822f.ipynb), dobra probka, ~30-35% kosztu pelnych 22 stacji.
#
# TEN notebook (jak 20260822c.ipynb) robi TYLKO wykrywanie cykli + liste
# zadan + WLASCIWY skan. Analiza/wykresy (trend korelacji vs P_days) sa w
# OSOBNYM notebooku (20260822h.ipynb) z tych samych powodow bezpieczenstwa
# co zawsze w tej sesji (przypadkowe ponowne odpalenie "od gory" nie moze
# wywolac ponownego kosztownego skanu).
#
# WAZNE: zapis do NOWEGO prefiksu plikow (results/pgrid_scan_*.csv), NIE
# cycle_scan_*.csv - zeby nie zmieszac z istniejacymi danymi (P_days=1675/
# 3350, 22 stacje) i NIE zepsuc zalozenia "2x2 P_days x d" w
# 20260822d.ipynb (ktory zaklada dokladnie 2 wartosci P_days).
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binom, norm
from scipy.signal import find_peaks
from statsmodels.nonparametric.smoothers_lowess import lowess

sys.path.insert(0, "..")
from mc_parallel import run_t0_scan_parallel

USGS_EXTENDED_PATH = "../data/usgs_data/usgs_m4_1965_2025.csv"

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_EXTENDED_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()


def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1
    Sm = sm_vals[1:]

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)
    sigma = norm.isf(pcdf)

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


eq = load_earthquakes(min_mag=4.0)
print(f"EQ (M>=4.0, katalog rozszerzony): {len(eq)} zdarzen, {eq.index.min()} .. {eq.index.max()}")


In [ ]:
# Loadery CR - TYLKO 5 stacji reprezentatywnych. mosc/oulu jak zawsze
# (pelna historia), athn/hrms/sopb z data/csv_data_stations_extended/
# (pobrane 22.08, pelna historia per stacja).
MOSC_PATH = "../data/mosc_data.csv"
OULU_PATH = "../data/oulu_5min_data.csv"
EXTENDED_DIR = "../data/csv_data_stations_extended"
REPRESENTATIVE_STATIONS = ["ATHN", "HRMS", "SOPB"]


def load_mosc():
    df = pd.read_csv(MOSC_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_oulu():
    df = pd.read_csv(OULU_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    n_bad = df["datetime"].isna().sum()
    if n_bad:
        print(f"load_oulu: pominieto {n_bad} niesparsowalnych wierszy (uszkodzone dane)")
    df = df.dropna(subset=["datetime"])
    return df.set_index("datetime").sort_index()["value"]


def load_station_extended(station):
    path = os.path.join(EXTENDED_DIR, f"{station.lower()}_extended_6h.csv")
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    return df.set_index("datetime").sort_index()["value"]


cr_series = {}
cr_series["mosc"] = load_mosc()
cr_series["oulu"] = load_oulu().resample("6h").mean()
for station in REPRESENTATIVE_STATIONS:
    cr_series[station.lower()] = load_station_extended(station)

for name, s in cr_series.items():
    print(f"{name}: {len(s)} pomiarow, {s.index.min()} .. {s.index.max()}")


In [ ]:
# Wykrywanie granic cykli slonecznych - identyczna logika co
# 20260822c.ipynb (LOWESS frac=0.01, find_peaks na odwroconym sygnale,
# distance=100, prominence=5).
sn = pd.read_csv("../data/SN_m_tot_V2.0.csv", sep=";")
sn.columns = [c.strip() for c in sn.columns]
sn["date"] = pd.to_datetime(dict(year=sn["rok"], month=sn["miesiac"], day=1))
smoothed = lowess(sn["sunspot"].values, sn["rok_norm"].values, frac=0.01)[:, 1]

min_idx, _ = find_peaks(-smoothed, distance=100, prominence=5)
cycle_min_dates = pd.to_datetime(sn["date"].values[min_idx])

cycles = []
for i in range(len(cycle_min_dates) - 1):
    start, end = cycle_min_dates[i], cycle_min_dates[i + 1]
    if end < eq.index.min() or start > eq.index.max():
        continue
    cycles.append(dict(label=f"cycle_{start.year}", start=max(start, eq.index.min()), end=min(end, eq.index.max())))

if len(cycle_min_dates) and cycle_min_dates[-1] <= eq.index.max():
    last_start = cycle_min_dates[-1]
    cycles.append(dict(label=f"cycle_{last_start.year}_incomplete", start=last_start, end=eq.index.max()))

print(f"{len(cycles)} segmentow cykli w zakresie katalogu EQ:")
for c in cycles:
    print(f"  {c['label']}: {c['start'].date()} .. {c['end'].date()} ({(c['end']-c['start']).days/365.25:.1f} lat)")


In [ ]:
# Parametry - D_VALUES/DT_DAYS/M/T0_STEP jak zawsze. P_DAYS_VALUES = TYLKO
# 6 NOWYCH wartosci (1675 i 3350 juz mamy policzone w cycle_scan_*.csv) -
# 3 pomiedzy 1675-3350 (rownomiernie), 3 powyzej 3350 w strone pelnej
# dlugosci cyklu (~11 lat = ok. 4015 dni, srednia z 5 wykrytych pelnych
# cykli w tym repo to ~10.9 roku - patrz 20260822c.ipynb komorka 2).
D_VALUES = [1, 5]
P_DAYS_VALUES = [2100, 2515, 2930, 3600, 3800, 4000]
DT_DAYS = 0
M_THRESHOLD = 4.0
T0_STEP = "24h"

print(f"D_VALUES={D_VALUES}, P_DAYS_VALUES={P_DAYS_VALUES}, T0_STEP={T0_STEP}")


In [ ]:
# Budowa listy zadan (stacja x cykl x P_days x d) + PODGLAD KOSZTU (na
# podstawie dzisiejszych rzeczywistych benchmarkow z 20260822c.ipynb:
# P=3350 d=1 ~61.6s/zadanie, d=5 ~13.7s/zadanie - skalowane liniowo po
# P_days, bo koszt cosmoseismic_stat rosnie z liczba binow = P_days/d).
tasks = []
for cyc in cycles:
    for P_days in P_DAYS_VALUES:
        t0_first = cyc["start"].ceil(T0_STEP)
        t0_last = min(cyc["end"], eq.index.max() - pd.Timedelta(days=P_days)).floor(T0_STEP)
        if t0_last <= t0_first:
            continue
        t0_candidates = pd.date_range(t0_first, t0_last, freq=T0_STEP)

        cr_lo, cr_hi = t0_first, t0_last + pd.Timedelta(days=P_days)
        for name, cr in cr_series.items():
            cr_cyc = cr[(cr.index >= cr_lo) & (cr.index <= cr_hi)]
            if len(cr_cyc) == 0:
                continue
            for d in D_VALUES:
                tasks.append(dict(
                    cycle=cyc["label"], station=name, P_days=P_days, d=d,
                    t0_candidates=t0_candidates, cr=cr_cyc,
                    n_candidates=len(t0_candidates),
                ))

print(f"Liczba zadan (stacja x cykl x P_days x d): {len(tasks)}")

preview = pd.DataFrame([{k: v for k, v in t.items() if k not in ("t0_candidates", "cr")} for t in tasks])
if len(preview):
    print("\nZadania per P_days/d:")
    print(preview.groupby(["P_days", "d"])["n_candidates"].agg(["count", "sum"]))

    BENCH_SEC_PER_CANDIDATE = {1: 61.6 / 4537, 5: 13.7 / 4537}  # z 20260822c.ipynb, cycle_1996 P=3350
    preview["est_sec"] = preview.apply(
        lambda r: r["n_candidates"] * BENCH_SEC_PER_CANDIDATE[r["d"]] * (r["P_days"] / 3350), axis=1
    )
    print("\nSzacowany czas per P_days (minuty):")
    print((preview.groupby("P_days")["est_sec"].sum() / 60).round(1))
    total_min = preview["est_sec"].sum() / 60
    print(f"\nZGRUBNY szacunek calkowitego czasu: {total_min:.0f} minut (~{total_min/60:.2f}h)")


In [ ]:
# WLASCIWY SKAN - kosztowna komorka (zgrubny szacunek ~2.8h, patrz komorka
# wyzej dla dokladnej liczby). Zapis do results/pgrid_scan_{stacja}_{cykl}_
# P{Pdays}_d{d}.csv - NOWY prefiks (nie cycle_scan_), zeby nie zmieszac z
# istniejacymi danymi 22-stacyjnymi i nie zepsuc zalozenia "2x2 P_days x d"
# w 20260822d.ipynb. Zapis ATOMOWY per zadanie + WZNAWIALNY (pomija juz
# istniejace pliki) - bezpiecznie przerywalny, tak jak 20260822c.ipynb.
RESULTS_DIR = "../results"
t_session_start = time.time()

for i, task in enumerate(tasks):
    out_path = f"{RESULTS_DIR}/pgrid_scan_{task['station']}_{task['cycle']}_P{task['P_days']}_d{task['d']}.csv"
    if os.path.exists(out_path):
        continue
    print(f"[{i+1}/{len(tasks)}] {task['station']} {task['cycle']} P={task['P_days']} d={task['d']} "
          f"({task['n_candidates']} kandydatow)...")
    t_task_start = time.time()
    run_t0_scan_parallel(
        task["cr"], eq, task["t0_candidates"],
        P_days=task["P_days"], d_days=task["d"], m=M_THRESHOLD, dt_days=DT_DAYS,
        stat_fn=cosmoseismic_stat,
        save_path=out_path,
    )
    elapsed_task = time.time() - t_task_start
    elapsed_total = (time.time() - t_session_start) / 60
    print(f"  gotowe w {elapsed_task:.1f}s (laczny czas tej sesji: {elapsed_total:.1f} min)")

print("\nWszystkie zadania ukonczone (albo juz wczesniej istnialy na dysku).")
